# Lesson 08 - Multi-Agent Design Pattern

## Setup

In [3]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [4]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Why Multi-Agent Systems?

Real-world tasks like trip planning involve many different kinds of expertise — logistics, local knowledge, budgeting, and more. A single agent trying to handle everything quickly becomes unwieldy.

Multi-agent systems solve this through **specialization**: each agent focuses on one area of expertise, producing higher-quality results than a generalist. They also improve **scalability** — you can add new agents (e.g., a flight specialist, a restaurant critic) without rewriting the existing workflow. The agents compose together through a structured pipeline, passing context from one to the next.

## Creating Specialized Agents

In [6]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Building a Sequential Workflow

`WorkflowBuilder` lets you wire agents into a directed graph. Here we create a simple two-step pipeline: the **TravelPlanner** drafts the itinerary, then the **TravelConcierge** reviews and enhances it.

In [7]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

C:\Users\lujan\AppData\Local\Temp\ipykernel_9148\1028293691.py:3: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()



🤖 TravelPlanner:
I can plan that — quick clarifying questions before I build the detailed 5‑day itinerary so it matches your needs:

1. Is the $3,000 total budget meant to include round‑trip flights? If so, where are you flying from (city/country)?
2. Travel dates or season (or flexible)?
3. Any dietary restrictions or strong dislikes (e.g., vegetarian, allergies)?
4. Preferred accommodation style: budget (hostel/B&B), mid‑range (3* hotel / well‑rated apartment), or upscale?
5. Pace: relaxed (more time to linger) or full (see/do a lot each day)?

If you prefer, I can give a ready‑to‑go sample itinerary assuming mid‑range lodging and that the $3,000 excludes flights — say you’re already in Europe — or an alternate version that includes flights from the U.S. Tell me which you want and I’ll produce the full day‑by‑day plan with costs, reservations tips, maps/transport advice, and where to book restaurants.

🤖 TravelConcierge:
Thanks — I’ve put together a ready-to-go 5‑day Paris plan for 

## Adding More Agents to the Workflow

One of the biggest advantages of the multi-agent pattern is how easy it is to extend. Below we add a **BudgetReviewer** agent that checks the plan against the traveler's budget, flags items that might push costs over the limit, and suggests money-saving alternatives. The workflow now runs three agents in sequence:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```

In [8]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

C:\Users\lujan\AppData\Local\Temp\ipykernel_9148\622960310.py:9: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()



🤖 TravelPlanner:
Quick question before I build the final, fully budgeted plan: does the $3,000 need to include round-trip airfare? And where are you flying from (city/country)? That will change recommendations a lot. If you want, I can proceed now with a sample itinerary that assumes airfare is NOT included and keep the $3,000 for accommodation, food, activities and local transport — or I can include flights if you tell me the departure city.

Below is a ready-to-use 5-day Paris itinerary tailored for a food-loving couple, built on the assumption that the $3,000 budget covers everything in Paris (accommodation, food, activities, local transit, some extras) but NOT international flights. If you want flights included, tell me your departure city and I’ll revise the budget and options.

Overview & budget guide (assumes 2 people, 5 days / 4 nights, excluding international airfare)
- Accommodation (mid-range apartment or 3-star hotel in central arrondissements): $120–$180/night → $480–$720

## Summary

In this lesson you learned how to:

1. **Create specialized agents** — each with a focused role (planning, concierge, budget review).
2. **Wire agents into a sequential workflow** using `WorkflowBuilder` and `add_edge`.
3. **Stream output** from a multi-agent pipeline, tracking which agent is speaking.
4. **Extend a workflow** by adding new agents to the chain without modifying existing ones.

The multi-agent design pattern keeps each agent simple while producing richer, more thoroughly reviewed results than any single agent could achieve alone.